In [9]:
### Econ 682: Time series
### Term Project
### By Collins Rostant


## Preparing Python to download series directly from FRED

!pip install pandas numpy matplotlib statsmodels pandas_datareader
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.api import VAR
from pandas_datareader import data as pdr
from statsmodels.regression.linear_model import OLS
from statsmodels.tools.tools import add_constant

## Downloading the series from FRED
start = '1990-01-01'
end = '2023-12-31'

series = {
    'CPI': 'CPIAUCSL',
    'UNRATE': 'UNRATE',
    'IP': 'INDPRO',
    'FFR': 'FEDFUNDS'
}

data = pd.DataFrame()

for name, code in series.items():
    data[name] = pdr.DataReader(code, 'fred', start, end)

data = data.dropna()

# Stationarity of series

# Inflation (log difference)
data['inflation'] = 100 * np.log(data['CPI']).diff()

# Industrial production growth
data['ip_growth'] = 100 * np.log(data['IP']).diff()

# Fed Funds Rate and Unemployment series are kept in levels

data = data.dropna()

df = data[['inflation', 'UNRATE', 'ip_growth', 'FFR']].copy()
df.columns = ['inflation', 'unemployment', 'ip', 'rate']

# Computing Covid pulse dummies

df['D_2020M3'] = (df.index == pd.Timestamp('2020-03-01')).astype(int)
df['D_2020M4'] = (df.index == pd.Timestamp('2020-04-01')).astype(int)
df['D_2020M5'] = (df.index == pd.Timestamp('2020-05-01')).astype(int)

# Plotting the time series

variables = {
    'inflation': 'Inflation (%)',
    'unemployment': 'Unemployment Rate (%)',
    'ip': 'Industrial Production Growth (%)',
    'rate': 'Federal Funds Rate (%)'
}

for var, title in variables.items():
    plt.figure()
    plt.plot(df.index, df[var])
    plt.axvline(pd.Timestamp('2020-03-01'), linestyle='--')  # COVID marker
    plt.title(title)
    plt.xlabel('Date')
    plt.ylabel(title)
    plt.tight_layout()
    plt.show()


# Summary statistics of the time series

summary = df[['inflation','unemployment','ip','rate']].describe().T[['mean','std','min','max']]
print(summary)


# VAR model

endog = df[['inflation', 'unemployment', 'ip', 'rate']]
exog = df[['D_2020M3', 'D_2020M4', 'D_2020M5']]

model = VAR(endog, exog=exog)

## Lag selection
lag_order = model.select_order(maxlags=12)
p = lag_order.selected_orders['bic']

print("Selected lag (BIC):", p)

# Estimating VAR model
var_model = model.fit(p)

print(var_model.summary())


# Impulse response functions

irf = var_model.irf(10)

irf.plot(orth=True)
plt.show()


# Estimating the pre-covid VAR with IRFs associated
df_pre = df[df.index < '2020-01-01']

model_pre = VAR(df_pre[['inflation','unemployment','ip','rate']])
p_pre = model_pre.select_order(12).selected_orders['bic']

var_pre = model_pre.fit(p_pre)

irf_pre = var_pre.irf(10)
irf_pre.plot(orth=True)
plt.show()

# Full vs Pre-Covid VAR comparison

horizon = 10
h = np.arange(horizon + 1)

# Compute IRFs
irf_full = var_model.irf(horizon)
irf_pre = var_pre.irf(horizon)

# Variables in order
var_names = ['inflation', 'unemployment', 'ip', 'rate']

# Comparing IRFs to a monetary policy shock (shock to 'rate')
shock_index = var_names.index('rate')

for response_var in ['inflation', 'unemployment', 'ip']:
    response_index = var_names.index(response_var)

    # Orthogonalized IRFs
    full_response = irf_full.orth_irfs[:, response_index, shock_index]
    pre_response = irf_pre.orth_irfs[:, response_index, shock_index]

    plt.figure()
    plt.plot(h, full_response, label='VAR Full Sample')
    plt.plot(h, pre_response, linestyle='--', label='VAR Pre-COVID')
    plt.axhline(0)

    plt.title(f'VAR Comparison: Response of {response_var} to a monetary policy shock')
    plt.xlabel('Horizon')
    plt.ylabel('Response')
    plt.legend()
    plt.tight_layout()
    plt.show()


    ## Local Projection analysis

    # Setting the LP function

def local_projection(df, shock_var, response_var, lags=4, horizon=10):
    betas = []
    lower = []
    upper = []

    for h in range(horizon + 1):

        df_lp = df.copy()

        # Future value Y_{t+h}
        df_lp['Y_future'] = df_lp[response_var].shift(-h)

        # Shock variable
        df_lp['shock'] = df_lp[shock_var]

        # Add lags
        for i in range(1, lags + 1):
            for col in ['inflation','unemployment','ip','rate']:
                df_lp[f'{col}_lag{i}'] = df_lp[col].shift(i)

        # COVID dummies
        df_lp['D_2020M3'] = df_lp['D_2020M3']
        df_lp['D_2020M4'] = df_lp['D_2020M4']
        df_lp['D_2020M5'] = df_lp['D_2020M5']

        df_lp = df_lp.dropna()

        X = df_lp[['shock'] +
                  [f'{col}_lag{i}' for col in ['inflation','unemployment','ip','rate'] for i in range(1,lags+1)] +
                  ['D_2020M3','D_2020M4','D_2020M5']]

        X = add_constant(X)
        Y = df_lp['Y_future']

        model = OLS(Y, X).fit(cov_type='HAC', cov_kwds={'maxlags': max(h,1)})

        beta = model.params['shock']
        se = model.bse['shock']

        betas.append(beta)
        lower.append(beta - 1.96*se)
        upper.append(beta + 1.96*se)

    return np.array(betas), np.array(lower), np.array(upper)

# Running LP for all series
    horizon = 10
lp_results = {}

for var in ['inflation','unemployment','ip']:
    betas, low, high = local_projection(df, 'rate', var, lags=4, horizon=horizon)
    lp_results[var] = (betas, low, high)

# Plotting IRFs for Local Projection
h = np.arange(horizon + 1)

for var in lp_results:
    betas, low, high = lp_results[var]

    plt.figure()
    plt.plot(h, betas, label='LP IRF')
    plt.fill_between(h, low, high, alpha=0.3)
    plt.axhline(0)

    plt.title(f'LP IRF: Response of {var}')
    plt.xlabel('Horizon')
    plt.ylabel('Response')
    plt.legend()
    plt.show()

# Analyzing LP pre-covid
df_pre = df[df.index < '2020-01-01']

lp_results_pre = {}

for var in ['inflation','unemployment','ip']:
    betas, low, high = local_projection(df_pre, 'rate', var, lags=4, horizon=horizon)
    lp_results_pre[var] = (betas, low, high)

# Full vs Pre-covid Local Projection Comparison

for var in ['inflation','unemployment','ip']:

    betas_full, _, _ = lp_results[var]
    betas_pre, _, _ = lp_results_pre[var]

    plt.figure()
    plt.plot(h, betas_full, label='LP Full Sample')
    plt.plot(h, betas_pre, linestyle='--', label='LP Pre-COVID')

    plt.axhline(0)
    plt.title(f'LP Comparison: {var}')
    plt.xlabel('Horizon')
    plt.ylabel('Response')
    plt.legend()
    plt.show()